# Agentic Competitor Analysis 🤖📊

Welcome to this interactive learning exercise! This notebook demonstrates how to build an **autonomous AI agent** that conducts deep competitive research on any company.

Here you will learn how to:
1. Use tools (web search, scrapers) with Python.
2. Expose these tools to an LLM.
3. Implement an Agent Loop that acts, thinks, and observes until it reaches a conclusion.
4. Format the final output into a professional markdown report.


## Step 1: Install Dependencies & Setup
We'll install the required libraries like `ddgs` (for searching), `trafilatura` (for scraping text), and `google-genai` (our brain).


In [ ]:
!pip install -q ddgs httpx trafilatura jinja2 pydantic google-genai nest_asyncio

# Let's create the folder structure we need
!mkdir -p output templates server/utils server/tools agent/llm


### Configure API Key
We're using **Google Gemini** for this tutorial (it's fast and has a great free tier). Get an API key at [Google AI Studio](https://aistudio.google.com/).


In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
except Exception:
    import getpass
    print("Please enter your Google Gemini API Key:")
    os.environ['GOOGLE_API_KEY'] = getpass.getpass()


## Step 2: The Utilities
Agents need "hands and eyes" to perceive the internet. We give them a `web_search` tool and a `scrape_urls` tool.


### Web Search Wrapper
Writing `server/utils/search.py` to the Colab environment.


In [ ]:
with open("server/utils/search.py", "w") as f:
    f.write(""""""DuckDuckGo search wrapper with rate limiting.

Provides a web_search() async function that wraps duckduckgo-search
with a token-bucket rate limiter and exponential backoff on rate limit errors.
"""

import asyncio
import logging
import os
import time

from ddgs import DDGS
from ddgs.exceptions import RatelimitException

logger = logging.getLogger(__name__)

_DEFAULT_MAX_RESULTS = int(os.getenv("MAX_SEARCH_RESULTS", "5"))
_DEFAULT_RATE_LIMIT = float(os.getenv("SEARCH_RATE_LIMIT", "1.0"))
_DEFAULT_MAX_RETRIES = int(os.getenv("SEARCH_MAX_RETRIES", "3"))
_DEFAULT_BACKOFF_BASE = float(os.getenv("SEARCH_BACKOFF_BASE", "2.0"))


class SearchRateLimiter:
    """Enforces minimum interval between search requests."""

    def __init__(self, min_interval: float = _DEFAULT_RATE_LIMIT):
        self._min_interval = min_interval
        self._lock = asyncio.Lock()
        self._last_call: float = 0.0

    async def execute(self, coro_func):
        async with self._lock:
            now = asyncio.get_event_loop().time()
            wait = self._min_interval - (now - self._last_call)
            if wait > 0:
                await asyncio.sleep(wait)
            result = await coro_func()
            self._last_call = asyncio.get_event_loop().time()
            return result


# Global rate limiter shared across all tool invocations in a single run
_rate_limiter = SearchRateLimiter()


async def web_search(
    query: str,
    max_results: int = _DEFAULT_MAX_RESULTS,
    max_retries: int = _DEFAULT_MAX_RETRIES,
    backoff_base: float = _DEFAULT_BACKOFF_BASE,
) -> list[dict]:
    """Search DuckDuckGo and return results.

    Args:
        query: The search query string.
        max_results: Maximum number of results to return.
        max_retries: Max retries on rate limit errors.
        backoff_base: Exponential backoff multiplier.

    Returns:
        A list of dicts with keys: title, href, body.
        Returns empty list on failure.
    """

    async def _do_search():
        ddgs = DDGS()
        return ddgs.text(query, max_results=max_results)

    for attempt in range(max_retries + 1):
        try:
            results = await _rate_limiter.execute(_do_search)
            return results if results else []
        except RatelimitException:
            if attempt < max_retries:
                wait = backoff_base ** attempt
                logger.warning(
                    "DuckDuckGo rate limit hit, retrying in %.1fs (attempt %d/%d)",
                    wait, attempt + 1, max_retries,
                )
                await asyncio.sleep(wait)
            else:
                logger.error("DuckDuckGo rate limit exhausted after %d retries", max_retries)
                return []
        except Exception:
            logger.exception("Unexpected error during web search for '%s'", query)
            return []

    return []
""")


### Web Content Scraper
Writing `server/utils/scraper.py` to the Colab environment.


In [ ]:
with open("server/utils/scraper.py", "w") as f:
    f.write(""""""Web content extractor using httpx + trafilatura.

Provides scrape_url() and scrape_multiple() async functions for
fetching and extracting readable content from web pages.
"""

import logging
import os

import httpx
import trafilatura

logger = logging.getLogger(__name__)

_TIMEOUT = int(os.getenv("REQUEST_TIMEOUT", "15"))


async def scrape_url(url: str, timeout: int = _TIMEOUT) -> str | None:
    """Fetch a URL and extract its main text content.

    Args:
        url: The URL to scrape.
        timeout: HTTP request timeout in seconds.

    Returns:
        Extracted text content, or None if extraction fails.
    """
    try:
        async with httpx.AsyncClient(
            timeout=timeout,
            follow_redirects=True,
            headers={"User-Agent": "Mozilla/5.0 (compatible; research-bot)"},
        ) as client:
            response = await client.get(url)

        if response.status_code != 200:
            logger.warning("Non-200 status %d for %s", response.status_code, url)
            return None

        content = trafilatura.extract(response.text)
        if not content:
            logger.debug("No content extracted from %s", url)
            return None

        return content

    except Exception:
        logger.exception("Failed to scrape %s", url)
        return None


async def scrape_multiple(urls: list[str], timeout: int = _TIMEOUT) -> list[str]:
    """Scrape multiple URLs and return successfully extracted content.

    Args:
        urls: List of URLs to scrape.
        timeout: HTTP request timeout in seconds per request.

    Returns:
        List of extracted text content (only successful extractions).
    """
    if not urls:
        return []

    results = []
    for url in urls:
        content = await scrape_url(url, timeout=timeout)
        if content:
            results.append(content)

    return results
""")


## Step 3: The Tools
Tools are Python functions that receive arguments, perform an action, and return JSON. The LLM decides *when* and *how* to call these.


### Tool: Validate Company
Writing `server/tools/validate_company.py` to the Colab environment.


In [ ]:
with open("server/tools/validate_company.py", "w") as f:
    f.write(""""""Company validation tool for the MCP server."""

import logging
from urllib.parse import urlparse

from server.utils.search import web_search

logger = logging.getLogger(__name__)

# Common non-company domains to skip when extracting the company domain
_SKIP_DOMAINS = {
    "wikipedia.org", "en.wikipedia.org",
    "crunchbase.com", "www.crunchbase.com",
    "linkedin.com", "www.linkedin.com",
    "bloomberg.com", "www.bloomberg.com",
    "forbes.com", "www.forbes.com",
    "twitter.com", "x.com",
    "youtube.com", "www.youtube.com",
    "github.com",
}


async def validate_company(company_name: str) -> dict:
    """Validate that a company is a real, identifiable entity.

    Searches the web for the given company name and confirms it exists.
    Returns the canonical company name, official domain, and a brief description.

    Args:
        company_name: The name of the company to validate (free-text input).

    Returns:
        A dict with keys:
            - name (str): Canonical company name.
            - domain (str): Official website domain.
            - description (str): Brief company description.
            - valid (bool): Whether the company was found.
            - error (str): Error message if validation failed.
    """
    try:
        results = await web_search(f"{company_name} company", max_results=5)
    except Exception as e:
        logger.exception("Search failed for company validation: %s", company_name)
        return {
            "name": company_name,
            "domain": "",
            "description": "",
            "valid": False,
            "error": f"Search failed: {e}",
        }

    if not results:
        return {
            "name": company_name,
            "domain": "",
            "description": "",
            "valid": False,
            "error": "No search results found.",
        }

    # Check if the company name appears in the search results
    company_lower = company_name.lower()
    name_found = any(
        company_lower in r.get("title", "").lower() or company_lower in r.get("body", "").lower()
        for r in results
    )

    if not name_found:
        return {
            "name": company_name,
            "domain": "",
            "description": "",
            "valid": False,
            "error": f"Could not confirm '{company_name}' as a real company.",
        }

    # Extract the official domain (first non-aggregator URL)
    domain = ""
    for r in results:
        href = r.get("href", "")
        parsed = urlparse(href)
        host = parsed.netloc.lower().removeprefix("www.")
        if host and host not in _SKIP_DOMAINS:
            domain = host
            break

    # Use the top result's body as the description
    description = results[0].get("body", "")

    return {
        "name": company_name,
        "domain": domain,
        "description": description,
        "valid": True,
    }
""")


### Tool: Identify Sector
Writing `server/tools/identify_sector.py` to the Colab environment.


In [ ]:
with open("server/tools/identify_sector.py", "w") as f:
    f.write(""""""Sector identification tool for the MCP server."""

import logging

from server.utils.search import web_search

logger = logging.getLogger(__name__)


async def identify_sector(company_name: str, domain: str) -> dict:
    """Identify the industry sector and sub-sector of a company.

    Searches the web for the company's industry classification and
    extracts sector information from the results.

    Args:
        company_name: The canonical name of the company.
        domain: The company's official website domain.

    Returns:
        A dict with keys:
            - sector (str): Primary industry sector.
            - sub_sector (str): More specific sub-sector.
            - sic_code (str): SIC code if found.
            - description (str): Brief sector description.
            - error (str): Error message if identification failed.
    """
    try:
        results = await web_search(
            f'"{company_name}" industry sector {domain}', max_results=5
        )
    except Exception as e:
        logger.exception("Search failed for sector identification: %s", company_name)
        return {
            "sector": "Unknown",
            "sub_sector": "",
            "sic_code": "",
            "description": "",
            "error": f"Search failed: {e}",
        }

    if not results:
        return {
            "sector": "Unknown",
            "sub_sector": "",
            "sic_code": "",
            "description": "",
        }

    # Combine all result bodies for analysis
    combined_text = " ".join(r.get("body", "") for r in results).lower()

    # Extract sector from results
    sector = _extract_sector(combined_text)
    sub_sector = _extract_sub_sector(combined_text, sector)
    sic_code = _extract_sic_code(combined_text)

    return {
        "sector": sector,
        "sub_sector": sub_sector,
        "sic_code": sic_code,
        "description": results[0].get("body", ""),
    }


def _extract_sector(text: str) -> str:
    """Extract the primary sector from combined search result text."""
    # Common sector keywords to look for
    sector_keywords = {
        "financial technology": "Financial Technology",
        "fintech": "Financial Technology",
        "artificial intelligence": "Artificial Intelligence",
        "cloud computing": "Cloud Computing",
        "e-commerce": "E-Commerce",
        "ecommerce": "E-Commerce",
        "cybersecurity": "Cybersecurity",
        "healthcare": "Healthcare",
        "health care": "Healthcare",
        "software": "Software",
        "saas": "Software as a Service",
        "telecommunications": "Telecommunications",
        "automotive": "Automotive",
        "retail": "Retail",
        "energy": "Energy",
        "media": "Media & Entertainment",
        "education": "Education",
        "real estate": "Real Estate",
        "logistics": "Logistics",
        "manufacturing": "Manufacturing",
        "biotechnology": "Biotechnology",
        "social media": "Social Media",
        "gaming": "Gaming",
        "food": "Food & Beverage",
        "travel": "Travel & Hospitality",
        "insurance": "Insurance",
        "banking": "Banking",
        "payments": "Financial Technology",
        "advertising": "Advertising Technology",
    }

    for keyword, sector in sector_keywords.items():
        if keyword in text:
            return sector

    return "Technology"


def _extract_sub_sector(text: str, sector: str) -> str:
    """Extract a sub-sector from the text, more specific than the primary sector."""
    sub_sector_keywords = {
        "payments": "Payments",
        "lending": "Lending",
        "analytics": "Analytics",
        "infrastructure": "Infrastructure",
        "marketplace": "Marketplace",
        "developer tools": "Developer Tools",
        "enterprise": "Enterprise Software",
        "consumer": "Consumer",
        "b2b": "B2B",
        "api": "API Platform",
        "data": "Data Services",
        "security": "Security",
        "automation": "Automation",
    }

    for keyword, sub in sub_sector_keywords.items():
        if keyword in text:
            return sub

    return ""


def _extract_sic_code(text: str) -> str:
    """Extract SIC code from the text if mentioned."""
    import re

    match = re.search(r"sic\s*(?:code)?\s*(\d{4})", text)
    return match.group(1) if match else ""
""")


### Tool: Find Competitors
Writing `server/tools/find_competitors.py` to the Colab environment.


In [ ]:
with open("server/tools/find_competitors.py", "w") as f:
    f.write(""""""Competitor discovery tool for the MCP server."""

import logging
import re
from urllib.parse import urlparse

from server.utils.search import web_search

logger = logging.getLogger(__name__)

_MAX_COMPETITORS = 3


async def find_competitors(company_name: str, sector: str) -> dict:
    """Find the top competitors of a company in its sector.

    Searches for direct competitors and cross-references with sector leaders.
    Returns up to 3 competitors with their domains and descriptions.

    Args:
        company_name: The canonical name of the company.
        sector: The company's industry sector.

    Returns:
        A dict with keys:
            - competitors (list[dict]): Up to 3 competitors, each with
              name, domain, and description.
            - error (str): Error message if discovery failed.
    """
    try:
        results = await web_search(
            f'"{company_name}" competitors alternatives', max_results=5
        )
    except Exception as e:
        logger.exception("Search failed for competitor discovery: %s", company_name)
        return {
            "competitors": [],
            "error": f"Search failed: {e}",
        }

    if not results:
        return {"competitors": []}

    # Extract competitor names from search results
    combined_text = " ".join(r.get("body", "") + " " + r.get("title", "") for r in results)
    competitors = _extract_competitors(combined_text, company_name)

    return {"competitors": competitors[:_MAX_COMPETITORS]}


def _extract_competitors(text: str, exclude_company: str) -> list[dict]:
    """Extract competitor information from combined search text."""
    # Common patterns in competitor articles
    # e.g., "competitors include Square, Adyen, and PayPal"
    # e.g., "Square (squareup.com) focuses on..."
    competitors = []
    seen_names = set()
    exclude_lower = exclude_company.lower()

    # Look for "Name (domain.com)" patterns
    domain_pattern = re.findall(
        r"(\b[A-Z][a-zA-Z]+(?:\s[A-Z][a-zA-Z]+)?)\s*\(([a-zA-Z0-9.-]+\.(?:com|io|net|org|co))\)",
        text,
    )
    for name, domain in domain_pattern:
        name_lower = name.lower()
        if name_lower != exclude_lower and name_lower not in seen_names:
            seen_names.add(name_lower)
            description = _find_description(text, name)
            competitors.append({
                "name": name,
                "domain": domain,
                "description": description,
            })

    # If we don't have enough from domain patterns, look for names after keywords
    if len(competitors) < _MAX_COMPETITORS:
        keyword_patterns = [
            r"competitors?\s*(?:include|are|:)\s*([^.]+)",
            r"(?:alternatives?\s*(?:to|include|are|:))\s*([^.]+)",
            r"(?:competes?\s*with)\s*([^.]+)",
        ]
        for pattern in keyword_patterns:
            matches = re.findall(pattern, text, re.IGNORECASE)
            for match in matches:
                # Split on commas and "and"
                names = re.split(r",\s*|\s+and\s+", match)
                for name in names:
                    name = name.strip().strip(".")
                    # Only accept proper-looking company names (capitalized, 1-3 words)
                    if (
                        name
                        and name[0].isupper()
                        and len(name.split()) <= 4
                        and name.lower() != exclude_lower
                        and name.lower() not in seen_names
                        and len(name) > 2
                    ):
                        seen_names.add(name.lower())
                        description = _find_description(text, name)
                        competitors.append({
                            "name": name,
                            "domain": "",
                            "description": description,
                        })

    return competitors


def _find_description(text: str, company_name: str) -> str:
    """Find a brief description of a company from the search text."""
    # Look for sentences containing the company name
    sentences = re.split(r"[.!?]+", text)
    for sentence in sentences:
        if company_name.lower() in sentence.lower() and len(sentence.strip()) > 20:
            desc = sentence.strip()
            # Limit to reasonable length
            if len(desc) > 200:
                desc = desc[:200].rsplit(" ", 1)[0] + "..."
            return desc
    return ""
""")


### Tool: Browse Company
Writing `server/tools/browse_company.py` to the Colab environment.


In [ ]:
with open("server/tools/browse_company.py", "w") as f:
    f.write(""""""Company research tool for the MCP server.

Browses the web for detailed information about a company across
multiple categories (pricing, products, marketing, market position).
"""

import logging
from datetime import datetime

from server.utils.search import web_search
from server.utils.scraper import scrape_multiple

logger = logging.getLogger(__name__)

_DEFAULT_CATEGORIES = ["pricing", "products", "marketing", "market_position"]
_TOP_URLS_PER_CATEGORY = 3

# Category-specific search query templates
_QUERY_TEMPLATES = {
    "pricing": '"{company}" pricing plans cost {domain} {year}',
    "products": '"{company}" products services features {domain}',
    "marketing": '"{company}" marketing strategy brand positioning',
    "market_position": '"{company}" market share position industry ranking',
}


def build_search_query(company_name: str, domain: str, category: str) -> str:
    """Build a targeted search query for a specific data category.

    Args:
        company_name: The company name.
        domain: The company's domain.
        category: One of pricing, products, marketing, market_position.

    Returns:
        A formatted search query string.
    """
    template = _QUERY_TEMPLATES.get(category, '"{company}" {category} {domain}')
    return template.format(
        company=company_name,
        domain=domain,
        category=category,
        year=datetime.now().year,
    )


async def browse_company(
    company_name: str,
    domain: str,
    categories: list[str] | None = None,
) -> dict:
    """Research a company across multiple data categories by searching and scraping the web.

    For each category, searches DuckDuckGo for targeted queries, scrapes the top
    results, and returns the extracted content. Handles partial failures gracefully.

    Args:
        company_name: The canonical name of the company.
        domain: The company's official website domain.
        categories: List of categories to research. Defaults to
            ["pricing", "products", "marketing", "market_position"].

    Returns:
        A dict with:
            - One key per category containing the extracted content (str).
            - categories_failed (list[str]): Categories that failed to retrieve data.
    """
    if categories is None:
        categories = list(_DEFAULT_CATEGORIES)

    result = {"categories_failed": []}

    for category in categories:
        try:
            content = await _research_category(company_name, domain, category)
            result[category] = content
        except Exception:
            logger.exception(
                "Failed to research %s for %s", category, company_name
            )
            result[category] = ""
            result["categories_failed"].append(category)

    return result


async def _research_category(
    company_name: str, domain: str, category: str
) -> str:
    """Research a single category for a company.

    Searches, scrapes top results, and combines content.
    Falls back to search snippets if scraping fails.
    """
    query = build_search_query(company_name, domain, category)
    search_results = await web_search(query, max_results=_TOP_URLS_PER_CATEGORY)

    if not search_results:
        return ""

    # Try to scrape the top URLs for detailed content
    urls = [r["href"] for r in search_results if r.get("href")]
    scraped_content = await scrape_multiple(urls[:_TOP_URLS_PER_CATEGORY])

    if scraped_content:
        return "\n\n---\n\n".join(scraped_content)

    # Fall back to search snippets
    snippets = [r.get("body", "") for r in search_results if r.get("body")]
    return "\n\n".join(snippets)
""")


### Tool: Generate Report
Writing `server/tools/generate_report.py` to the Colab environment.


In [ ]:
with open("server/tools/generate_report.py", "w") as f:
    f.write(""""""Report generation tool for the MCP server."""

import logging
import os
from datetime import datetime
from types import SimpleNamespace

import jinja2

logger = logging.getLogger(__name__)

_OUTPUT_DIR = os.getenv("OUTPUT_DIR", "output")
_TEMPLATE_DIR = os.path.join(os.path.dirname(os.path.dirname(os.path.dirname(__file__))), "templates")


from typing import Any

async def generate_report(
    target_company: dict,
    sector: dict,
    competitors: list[Any],
    target_data: dict,
    executive_summary: str = "",
    swot: dict | None = None,
    recommendations: str = "",
) -> dict:
    """Generate a competitive analysis markdown report.

    Renders a Jinja2 template with all collected data and saves it
    to the output directory with a timestamped filename.

    Args:
        target_company: Dict with name, domain, description of the target.
        sector: Dict with sector, sub_sector.
        competitors: List of competitor dicts with name, domain, description, data.
        target_data: Dict with pricing, products, marketing, market_position.
        executive_summary: Executive summary text.
        swot: Dict with strengths, weaknesses, opportunities, threats.
        recommendations: Recommendations text.

    Returns:
        A dict with keys:
            - report_path (str): Path to the saved report file.
            - summary (str): Brief summary of the report.
    """
    generated_at = datetime.now().strftime("%Y-%m-%d %H:%M")
    date_slug = datetime.now().strftime("%Y-%m-%d")

    # Make competitor data accessible via dot notation in template
    for comp in competitors:
        if "data" in comp and isinstance(comp["data"], dict):
            comp["data"] = SimpleNamespace(**comp["data"])

    # Collect failed categories
    categories_failed = []
    if isinstance(target_data, dict):
        target_data_ns = SimpleNamespace(**target_data)
    else:
        target_data_ns = target_data

    if swot and isinstance(swot, dict):
        swot = SimpleNamespace(**swot)

    # Load and render template
    env = jinja2.Environment(
        loader=jinja2.FileSystemLoader(_TEMPLATE_DIR),
        undefined=jinja2.Undefined,
    )
    template = env.get_template("report.md.j2")

    report_content = template.render(
        target=SimpleNamespace(**target_company),
        sector=SimpleNamespace(**sector),
        competitors=competitors,
        target_data=target_data_ns,
        executive_summary=executive_summary or "No executive summary provided.",
        swot=swot,
        recommendations=recommendations,
        generated_at=generated_at,
        categories_failed=categories_failed,
    )

    # Save report
    company_slug = target_company["name"].lower().replace(" ", "_")
    filename = f"{company_slug}_{date_slug}.md"
    report_path = os.path.join(_OUTPUT_DIR, filename)

    os.makedirs(_OUTPUT_DIR, exist_ok=True)

    with open(report_path, "w") as f:
        f.write(report_content)

    # Generate brief summary
    summary = (
        f"Competitive analysis report for {target_company['name']} saved to {report_path}. "
        f"Analyzed {len(competitors)} competitors in the {sector.get('sector', 'Unknown')} sector."
    )

    return {
        "report_path": report_path,
        "summary": summary,
    }
""")


### Report Markdown Template
Writing `templates/report.md.j2` to the Colab environment.


In [ ]:
with open("templates/report.md.j2", "w") as f:
    f.write("""# Competitive Analysis Report: {{ target.name }}

**Generated**: {{ generated_at }}

---

## Executive Summary

{{ executive_summary }}

---

## Company Overview

**Company**: {{ target.name }}
**Domain**: {{ target.domain }}
**Sector**: {{ sector.sector }}{% if sector.sub_sector %} / {{ sector.sub_sector }}{% endif %}

{{ target.description }}

---

## Competitor Profiles

{% for comp in competitors %}
### {{ loop.index }}. {{ comp.name }}

**Domain**: {{ comp.domain }}

{{ comp.description }}

{% if comp.data %}
{% if comp.data.pricing %}
**Pricing**: {{ comp.data.pricing[:500] }}
{% endif %}
{% if comp.data.products %}
**Products**: {{ comp.data.products[:500] }}
{% endif %}
{% endif %}

{% endfor %}

---

## Comparison Matrix

| Category | {{ target.name }} | {% for comp in competitors %}{{ comp.name }} | {% endfor %}

|----------|{{ '-' * (target.name | length + 2) }}|{% for comp in competitors %}{{ '-' * (comp.name | length + 2) }}|{% endfor %}

{% if target_data.pricing %}| **Pricing** | {{ target_data.pricing[:100] | replace('\n', ' ') }} | {% for comp in competitors %}{{ (comp.data.pricing[:100] if comp.data and comp.data.pricing else 'N/A') | replace('\n', ' ') }} | {% endfor %}
{% endif %}
{% if target_data.products %}| **Products** | {{ target_data.products[:100] | replace('\n', ' ') }} | {% for comp in competitors %}{{ (comp.data.products[:100] if comp.data and comp.data.products else 'N/A') | replace('\n', ' ') }} | {% endfor %}
{% endif %}
{% if target_data.market_position %}| **Market Position** | {{ target_data.market_position[:100] | replace('\n', ' ') }} | {% for comp in competitors %}{{ (comp.data.market_position[:100] if comp.data and comp.data.market_position else 'N/A') | replace('\n', ' ') }} | {% endfor %}
{% endif %}

---

## SWOT Analysis: {{ target.name }}

### Strengths
{{ swot.strengths if swot and swot.strengths else 'Data not available.' }}

### Weaknesses
{{ swot.weaknesses if swot and swot.weaknesses else 'Data not available.' }}

### Opportunities
{{ swot.opportunities if swot and swot.opportunities else 'Data not available.' }}

### Threats
{{ swot.threats if swot and swot.threats else 'Data not available.' }}

---

## Recommendations

{{ recommendations if recommendations else 'No recommendations generated.' }}

---

## Data Sources & Methodology

- **Search Engine**: DuckDuckGo
- **Data Collection Date**: {{ generated_at }}
- **Categories Analyzed**: Pricing, Products, Marketing, Market Position
- **Methodology**: Automated web search and content extraction

{% if categories_failed %}
**Note**: The following data categories could not be fully retrieved: {{ categories_failed | join(', ') }}
{% endif %}

---

*Report generated by Competitive Analysis AI Agent*
""")


## Step 4: The LLM Brain
Here we define a `GeminiClient` that wraps `google-genai` to easily chat with the model and extract the tools it wants to call.


### LLM Target Interfaces
Writing `agent/llm/base.py` to the Colab environment.


In [ ]:
with open("agent/llm/base.py", "w") as f:
    f.write(""""""Abstract base class for LLM client implementations.

Each provider (Gemini, Anthropic, OpenAI) implements this interface.
The agent loop uses only these methods, making it provider-agnostic.
"""

from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Any


@dataclass
class ToolCall:
    """Normalized tool call extracted from an LLM response."""
    id: str
    name: str
    arguments: dict


@dataclass
class ToolDefinition:
    """MCP tool definition converted to a provider-agnostic format."""
    name: str
    description: str
    parameters: dict  # JSON Schema


class LLMClient(ABC):
    """Abstract LLM client interface.

    Implementations normalize provider-specific formats into common
    ToolCall/ToolDefinition structures so the agent loop doesn't need
    to know which provider is active.
    """

    @abstractmethod
    async def chat(
        self,
        messages: list[dict],
        tools: list[ToolDefinition],
        system_prompt: str = "",
    ) -> Any:
        """Send messages to the LLM and get a response.

        Args:
            messages: Conversation history in provider-neutral format.
            tools: Available tool definitions.
            system_prompt: System prompt for the conversation.

        Returns:
            Provider-specific response object.
        """

    @abstractmethod
    def extract_tool_calls(self, response: Any) -> list[ToolCall]:
        """Extract tool calls from a provider response.

        Args:
            response: Provider-specific response object.

        Returns:
            List of normalized ToolCall objects. Empty if no tool calls.
        """

    @abstractmethod
    def extract_text(self, response: Any) -> str:
        """Extract text content from a provider response.

        Args:
            response: Provider-specific response object.

        Returns:
            Text content from the response.
        """

    @abstractmethod
    def format_tool_result(self, tool_call: ToolCall, result: str) -> dict:
        """Format a tool result for the provider's expected message format.

        Args:
            tool_call: The original tool call.
            result: The tool's output as a string.

        Returns:
            A message dict in the provider's expected format.
        """

    @abstractmethod
    def format_assistant_message(self, response: Any) -> dict:
        """Convert a provider response to a message dict for conversation history.

        Args:
            response: Provider-specific response object.

        Returns:
            A message dict to append to conversation history.
        """

    @abstractmethod
    def convert_tool_definitions(self, tools: list[ToolDefinition]) -> list[Any]:
        """Convert generic tool definitions to provider-specific format.

        Args:
            tools: List of provider-agnostic tool definitions.

        Returns:
            List of tool definitions in the provider's expected format.
        """
""")


### Gemini Integration
Writing `agent/llm/gemini_client.py` to the Colab environment.


In [ ]:
with open("agent/llm/gemini_client.py", "w") as f:
    f.write(""""""Gemini LLM client implementation using google-genai SDK."""

import json
import os
from typing import Any

from google import genai
from google.genai import types

from agent.llm.base import LLMClient, ToolCall, ToolDefinition

_DEFAULT_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")


class GeminiClient(LLMClient):
    """LLM client for Google Gemini."""

    def __init__(self, api_key: str, model: str = _DEFAULT_MODEL):
        self._client = genai.Client(api_key=api_key)
        self._model = model

    async def chat(
        self,
        messages: list[dict],
        tools: list[ToolDefinition],
        system_prompt: str = "",
    ) -> Any:
        gemini_tools = self.convert_tool_definitions(tools)

        config = types.GenerateContentConfig(
            system_instruction=system_prompt if system_prompt else None,
            tools=[types.Tool(function_declarations=gemini_tools)] if gemini_tools else None,
        )

        contents = self._convert_messages(messages)

        response = self._client.models.generate_content(
            model=self._model,
            contents=contents,
            config=config,
        )
        return response

    def extract_tool_calls(self, response: Any) -> list[ToolCall]:
        calls = []
        for candidate in response.candidates:
            if not candidate.content or not candidate.content.parts:
                continue
            for part in candidate.content.parts:
                if part.function_call:
                    fc = part.function_call
                    args = dict(fc.args) if fc.args else {}
                    call_id = fc.id if hasattr(fc, "id") and fc.id else f"gemini_{fc.name}"
                    calls.append(ToolCall(id=call_id, name=fc.name, arguments=args))
        return calls

    def extract_text(self, response: Any) -> str:
        try:
            return response.text or ""
        except (AttributeError, ValueError):
            return ""

    def format_tool_result(self, tool_call: ToolCall, result: str) -> dict:
        return {
            "role": "tool",
            "parts": [
                types.Part.from_function_response(
                    name=tool_call.name,
                    response={"result": result},
                )
            ],
        }

    def format_assistant_message(self, response: Any) -> dict:
        parts = []
        if response.candidates and response.candidates[0].content and response.candidates[0].content.parts:
            parts = response.candidates[0].content.parts
            
        return {
            "role": "model",
            "parts": parts,
        }

    def convert_tool_definitions(self, tools: list[ToolDefinition]) -> list[Any]:
        def fix_schema(schema: dict) -> dict:
            """Recursively fix JSON schema for Gemini compatibility."""
            if not isinstance(schema, dict):
                return schema
            
            # Make a copy to avoid mutating the original
            fixed = schema.copy()
            
            # Gemini models require 'items' if type is array
            if fixed.get("type") == "array" and "items" not in fixed:
                fixed["items"] = {"type": "object"}  # Default to object types if unspecified
                
            for k, v in fixed.items():
                if isinstance(v, dict):
                    fixed[k] = fix_schema(v)
                elif isinstance(v, list):
                    fixed[k] = [fix_schema(item) if isinstance(item, dict) else item for item in v]
                    
            return fixed

        declarations = []
        for tool in tools:
            params = dict(tool.parameters) if tool.parameters else {}
            params = fix_schema(params)
            params.pop("additionalProperties", None)

            declarations.append(
                types.FunctionDeclaration(
                    name=tool.name,
                    description=tool.description,
                    parameters=params if params.get("properties") else None,
                )
            )
        return declarations

    def _convert_messages(self, messages: list[dict]) -> list[Any]:
        """Convert provider-neutral messages to Gemini format."""
        contents = []
        for msg in messages:
            role = "model" if msg["role"] == "assistant" else msg["role"]
            if "parts" in msg:
                # If it's already structured, unpack it
                parts = []
                for p in msg["parts"]:
                    # check if p is already a Part
                    if isinstance(p, types.Part):
                        parts.append(p)
                    elif isinstance(p, dict) and "text" in p:
                        parts.append(types.Part.from_text(text=p["text"]))
                    elif isinstance(p, str):
                        parts.append(types.Part.from_text(text=p))
                    else:
                        parts.append(p)
                contents.append(types.Content(role=role, parts=parts))
            else:
                contents.append(
                    types.Content(role=role, parts=[types.Part.from_text(text=msg.get("content", ""))])
                )
        return contents
""")


## Step 5: The Agent Loop
This is the orchestrator! It sits in a `WHILE` loop, pushing tool executions back to the model until the model successfully generates a report.


### Agent Orchestrator
Writing `agent/client.py` to the Colab environment.


In [ ]:
with open("agent/client.py", "w") as f:
    f.write(""""""Agent loop — provider-agnostic orchestrator.

Connects to the MCP server, discovers tools, and runs a conversation
loop with the configured LLM provider until the agent produces a
final text response.
"""

import json
import logging

from agent.llm.base import LLMClient, ToolCall, ToolDefinition

logger = logging.getLogger(__name__)

_SYSTEM_PROMPT = """You are a competitive analysis agent. Given a company name, you must:

1. Call validate_company to confirm the company exists and get its domain.
2. Call identify_sector to determine the company's industry sector.
3. Call find_competitors to discover the top 3 competitors.
4. Call browse_company for the target company to research pricing, products, marketing, and market position.
5. Call browse_company for each competitor (same categories).
6. Call generate_report with all collected data to produce the final report.

Important rules:
- Always start with validate_company. If it returns valid: false, inform the user and stop.
- Use the exact tool names and parameter formats specified.
- When calling generate_report, include an executive_summary, swot analysis, and recommendations based on your analysis of the collected data.
- Be thorough but concise in your analysis.
- If a tool call fails, continue with the data you have.
"""

_DEFAULT_MAX_ITERATIONS = 20


class AgentLoop:
    """Provider-agnostic agent orchestration loop.

    Takes an LLMClient and a list of tool definitions, then runs a
    conversation loop: send messages to the LLM, execute any tool calls,
    and repeat until the LLM returns a final text response.
    """

    def __init__(
        self,
        llm_client: LLMClient,
        tools: list[ToolDefinition],
        max_iterations: int = _DEFAULT_MAX_ITERATIONS,
        tool_executor=None,
    ):
        self._llm = llm_client
        self._tools = tools
        self._max_iterations = max_iterations
        self._tool_executor = tool_executor

    async def run(self, user_message: str) -> str:
        """Run the agent loop with the given user message.

        Args:
            user_message: The user's input (e.g., a company name).

        Returns:
            The final text response from the LLM.
        """
        messages = [{"role": "user", "content": user_message}]

        for iteration in range(self._max_iterations):
            logger.info("Agent iteration %d/%d", iteration + 1, self._max_iterations)

            response = await self._llm.chat(
                messages=messages,
                tools=self._tools,
                system_prompt=_SYSTEM_PROMPT,
            )

            tool_calls = self._llm.extract_tool_calls(response)

            if not tool_calls:
                return self._llm.extract_text(response)

            # Append the assistant's response to conversation history
            assistant_msg = self._llm.format_assistant_message(response)
            messages.append(assistant_msg)

            # Execute each tool call and append results
            for tc in tool_calls:
                logger.info("Executing tool: %s(%s)", tc.name, json.dumps(tc.arguments)[:200])

                try:
                    result = await self._execute_tool(tc)
                except Exception as e:
                    logger.error("Tool %s failed: %s", tc.name, e)
                    result = json.dumps({"error": str(e)})

                tool_result_msg = self._llm.format_tool_result(tc, result)
                messages.append(tool_result_msg)

        # Max iterations reached
        logger.warning("Agent reached max iterations (%d)", self._max_iterations)
        return self._llm.extract_text(response)

    async def _execute_tool(self, tool_call: ToolCall) -> str:
        """Execute a tool call via the MCP server or direct function call.

        Args:
            tool_call: The normalized tool call to execute.

        Returns:
            The tool's result as a JSON string.
        """
        if self._tool_executor:
            result = await self._tool_executor(tool_call.name, tool_call.arguments)
        else:
            # Import and call the tool function directly
            result = await self._call_tool_directly(tool_call)

        if isinstance(result, dict):
            return json.dumps(result)
        return str(result)

    async def _call_tool_directly(self, tool_call: ToolCall) -> dict:
        """Call a tool function directly by name."""
        from server.tools.validate_company import validate_company
        from server.tools.identify_sector import identify_sector
        from server.tools.find_competitors import find_competitors
        from server.tools.browse_company import browse_company
        from server.tools.generate_report import generate_report

        tool_map = {
            "validate_company": validate_company,
            "identify_sector": identify_sector,
            "find_competitors": find_competitors,
            "browse_company": browse_company,
            "generate_report": generate_report,
        }

        func = tool_map.get(tool_call.name)
        if not func:
            return {"error": f"Unknown tool: {tool_call.name}"}

        return await func(**tool_call.arguments)
""")


## Step 6: Bring It All Together
Here we wire up the files, select the company we want to research, and kick off the agent!


In [ ]:
import nest_asyncio
import asyncio
from IPython.display import display, Markdown

# Allow running asyncio in Colab
nest_asyncio.apply()

# Initialize LLM and Agent
from agent.client import Agent
from agent.llm.gemini_client import GeminiClient

async def run_analysis(company_name: str):
    print(f"\n🚀 Starting Analysis for: {company_name}")
    print("-" * 40)
    
    llm = GeminiClient(api_key=os.environ["GOOGLE_API_KEY"])
    agent = Agent(llm=llm, max_iterations=20)
    
    # Run loop
    try:
        result = await agent.run(f"Perform a competitive analysis of {company_name} and generate the report.")
        print("\n✅ Analysis complete!")
        
        # Display the result (fallback to text if the tool didn't return report_path explicitly directly to agent)
        if hasattr(result, "get") and result.get("report_path"):
            filename = result["report_path"]
        else:
            # guess the filename based on timestamp since the generate_report tool saves it to output/
            import os, glob
            files = glob.glob("output/*.md")
            filename = max(files, key=os.path.getctime) if files else None
            
        if filename:
            with open(filename, "r") as f:
                display(Markdown(f.read()))
        else:
            print("Could not find the generated report file.")
            
    except Exception as e:
        print(f"\n❌ Error during execution: {e}")

# ==========================================
# CHANGE THIS TO ANY COMPANY YOU WANT!
# ==========================================
await run_analysis("Anthropic")

